# New Cairo Apartment Advisor

Cleaning, analysis, and model training for sale and rent apartment pricing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

raw_path = Path("../data/raw_selected_columns.csv")
sale_path = Path("../data/sale_apartments_clean.csv")
rent_path = Path("../data/rent_apartments_clean.csv")

sale = pd.read_csv(sale_path)
rent = pd.read_csv(rent_path)
print("Sale rows:", len(sale))
print("Rent rows:", len(rent))
sale.head()

## Market summary

In [ ]:
summary = pd.DataFrame({
    "mode": ["sale", "rent"],
    "rows": [len(sale), len(rent)],
    "median_price": [sale.price.median(), rent.price.median()],
    "median_price_per_m2": [sale.price_per_m2.median(), rent.price_per_m2.median()],
    "mean_price_per_m2": [sale.price_per_m2.mean(), rent.price_per_m2.mean()]
})
summary

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(sale.price_per_m2, bins=30)
plt.title("Sale price per m² distribution")
plt.xlabel("EGP per m²")
plt.ylabel("Listings")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(rent.price_per_m2, bins=30)
plt.title("Rent price per m² distribution")
plt.xlabel("EGP per m² per month")
plt.ylabel("Listings")
plt.show()

## Models

Two models are trained and saved in `/models`: sale and rent. The app uses these saved models.

In [ ]:
import json
with open("../reports/model_metrics.json") as f:
    metrics = json.load(f)
metrics

## Example prediction

In [ ]:
import joblib
model = joblib.load("../models/sale_price_model.joblib")
example = pd.DataFrame([{
    "area_m2": 160, "bedrooms": 3, "bathrooms": 2,
    "latitude": 30.04, "longitude": 31.47, "furnished": 0,
    "district": "The 5th Settlement", "payment_plan": "Unknown"
}])
predicted_value = model["model"].predict(example)[0]
pred = predicted_value * example.loc[0, "area_m2"] if model.get("target") == "price_per_m2" else predicted_value
print(f"Estimated sale price: {pred:,.0f} EGP")